# Embeddings and vector stores

Exercise notebook for the lesson **[Embeddings and vector stores](https://ramkumarmd.com/medical-ai/rag/embeddings-and-vector-stores)** in the *Retrieval-augmented generation (RAG)* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** An embedding model turns text into a list of numbers so that passages with similar meaning have similar vectors; a vector store keeps these vectors and finds the closest ones to a question. Embeddings find matches that keyword search misses, such as 'weak kidneys' for a section on kidney function, but a general-purpose model can still miss clinical abbreviations such as 'hypo'. Abbreviation expansion, hybrid keyword-plus-embedding search and domain-specific models help, and every choice should be checked with evaluation.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Keyword and embedding retrieval side by side

Run the protocol block from the chunking lesson first. This splits it into sections.

In [ ]:
import re

sections = [p.strip() for p in re.split(r"\n(?=## )", PROTOCOL)[1:]]
titles = [s.splitlines()[0].lstrip("# ") for s in sections]
print(titles)

# ['1. Monitoring glucose control', '2. Kidney function and metformin', '3. Foot care', '4. Low blood glucose', '5. Eye screening']

### Six patient- and clinician-style questions

Continuing from the previous block: best-matching section by TF-IDF and by embeddings.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

questions = ["My sugar dropped really low, what now?", "What should I do about a hypo?",
             "Is the diabetes pill safe with weak kidneys?", "When do we need to stop the biguanide?",
             "How often are retinal photos needed?", "Numb toes test"]

vec = TfidfVectorizer(stop_words="english").fit(sections + questions)
kw = cosine_similarity(vec.transform(questions), vec.transform(sections))
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
em = model.encode(questions, normalize_embeddings=True) @ model.encode(sections, normalize_embeddings=True).T
for q, k, e in zip(questions, kw, em):
    print(f"{q:<45} keyword: {titles[k.argmax()][:24]:<25} ({k.max():.2f})  "
          f"embedding: {titles[e.argmax()][:24]:<25} ({e.max():.2f})")

# My sugar dropped really low, what now?        keyword: 4. Low blood glucose      (0.17)  embedding: 4. Low blood glucose      (0.56)
# What should I do about a hypo?                keyword: 1. Monitoring glucose co  (0.00)  embedding: 3. Foot care              (0.24)
# Is the diabetes pill safe with weak kidneys?  keyword: 5. Eye screening          (0.08)  embedding: 2. Kidney function and m  (0.42)
# When do we need to stop the biguanide?        keyword: 2. Kidney function and m  (0.08)  embedding: 2. Kidney function and m  (0.16)
# How often are retinal photos needed?          keyword: 5. Eye screening          (0.10)  embedding: 5. Eye screening          (0.36)
# Numb toes test                                keyword: 3. Foot care              (0.09)  embedding: 3. Foot care              (0.53)

### Helping retrieval with clinical abbreviations

Continuing from the previous blocks: expand abbreviations, then search with embeddings.

In [ ]:
ABBREVIATIONS = {"hypo": "hypo (hypoglycaemia, low blood glucose)",
                 "biguanide": "biguanide (metformin)"}

def expand(question):
    for short, long in ABBREVIATIONS.items():
        question = re.sub(rf"\b{short}\b", long, question, flags=re.IGNORECASE)
    return question

section_emb = model.encode(sections, normalize_embeddings=True)
for q in ["What should I do about a hypo?", "When do we need to stop the biguanide?"]:
    for label, text in [("original", q), ("expanded", expand(q))]:
        scores = model.encode([text], normalize_embeddings=True) @ section_emb.T
        print(f"{label:<9} {text}\n          -> {titles[scores.argmax()]} ({scores.max():.2f})")

# original  What should I do about a hypo?
#           -> 3. Foot care (0.24)
# expanded  What should I do about a hypo (hypoglycaemia, low blood glucose)?
#           -> 4. Low blood glucose (0.61)
# original  When do we need to stop the biguanide?
#           -> 2. Kidney function and metformin (0.16)
# expanded  When do we need to stop the biguanide (metformin)?
#           -> 2. Kidney function and metformin (0.42)

### A minimal vector store

Continuing from the previous blocks: store sections with metadata and search with a version filter.

In [ ]:
import numpy as np

class TinyVectorStore:
    """Holds normalised embeddings with their text and metadata; finds the nearest by cosine similarity."""
    def __init__(self, model):
        self.model, self.vectors, self.records = model, None, []

    def add(self, texts, metadata):
        emb = self.model.encode(texts, normalize_embeddings=True)
        self.vectors = emb if self.vectors is None else np.vstack([self.vectors, emb])
        self.records += [{"text": t, **m} for t, m in zip(texts, metadata)]

    def search(self, query, k=2, **filters):
        scores = self.vectors @ self.model.encode([query], normalize_embeddings=True)[0]
        hits = [(s, r) for s, r in zip(scores, self.records)
                if all(r.get(key) == value for key, value in filters.items())]
        return sorted(hits, key=lambda h: -h[0])[:k]

store = TinyVectorStore(model)
store.add(sections, [{"source": "clinic-protocol", "version": "2026", "section": t} for t in titles])
for score, record in store.search("Is the diabetes pill safe with weak kidneys?", k=2, version="2026"):
    print(f"{score:.2f}  {record['section']}  (source {record['source']} v{record['version']})")
print(f"Stored {store.vectors.shape[0]} vectors of {store.vectors.shape[1]} dimensions")

# 0.42  2. Kidney function and metformin  (source clinic-protocol v2026)
# 0.35  4. Low blood glucose  (source clinic-protocol v2026)
# Stored 5 vectors of 384 dimensions

## Exercise

**Goal:** Compare keyword and embedding retrieval, improve abbreviation handling and build a filtered vector store.

**Data:** Synthetic protocol in the notebook; small open embedding model all-MiniLM-L6-v2 (Course content; model under Apache 2.0)

### Step 1

Reproduce the six-question comparison and add six questions of your own, including two abbreviations and two drug-class names.

In [ ]:
# Your code here


### Step 2

Implement BM25 (or use the rank_bm25 package) and add it to the comparison.

In [ ]:
# Your code here


### Step 3

Extend the abbreviation dictionary and measure how many of your questions it fixes, and whether any get worse.

In [ ]:
# Your code here


### Step 4

Add the outdated 2024 protocol to the vector store with version '2024' and show that the version filter excludes it.

In [ ]:
# Your code here


### Step 5

Optional: try a medical embedding model (for example MedCPT) and compare results on the same questions.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/rag/embeddings-and-vector-stores](https://ramkumarmd.com/medical-ai/rag/embeddings-and-vector-stores). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.